| Encoder | Logic | Best Use Case |
| :--- | :--- | :--- |
| **One-Hot** | Creates a new binary column (0 or 1) for every unique category in the feature. | **Low Cardinality:** Ideal for features with few unique values (e.g., "Color" or "Department"). |
| **Hash** | Uses a mathematical function to map categories into a fixed number of "buckets" or columns. | **High Cardinality/Streaming:** Best for massive datasets where you must cap memory usage. |
| **Target** | Replaces a category with the (blended) mean of the target variable for that group. | **High Cardinality/High Signal:** Best when the category name itself directly correlates to the outcome. |

---

### Other Encoders

*   **Label / Ordinal:** Assigns a unique integer to each category; best for tree-based models or data with a natural rank (e.g., "Bronze", "Silver", "Gold").
*   **Binary:** Converts categories into binary code and splits the digits into a small number of columns to balance memory and information.
*   **Count / Frequency:** Replaces the category name with its total number of occurrences to capture "popularity" as a numerical signal.
*   **Leave-One-Out:** A variation of Target Encoding that calculates the mean excluding the current row's value to further reduce data leakage.

In [2]:
# loader scripts

import requests
import pandas as pd
import numpy as np

def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
    .groupby(tz_offset)
    [time_col]
    .transform(lambda s: pd.to_datetime(s)
    .dt.tz_localize(s.name, ambiguous=True)
    .dt.tz_convert(tz_name))
    )



def load_fuel_economy_data() -> pd.DataFrame :
    url = 'https://www.fueleconomy.gov/feg/epadata/vehicles.csv'
    cols = ['year', 'make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr',
            'barrels08', 'city08', 'comb08', 'range', 'evMotor', 'cylinders', 'displ', 'fuelCost08',
            'fuelType', 'highway08', 'trans_dscr','createdOn']

    raw =  pd.read_csv(url)
    autos = (raw
        .loc[:, cols]
        .assign(
            # Extract Timezone Offset (e.g., EDT -> EST5EDT)
            offset=(raw.createdOn.str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                    ['offset'].replace('EDT', 'EST5EDT')),

            # Reconstruct date string (removing the original TZ name for parsing)
            str_date=(raw.createdOn.str.slice(4, 19) + " " +
                    raw.createdOn.str.slice(-4)),

            # Convert to localized datetime
            createdOn=lambda df_: to_tz(df_, 'str_date', 'offset', 'America/New_York')
        )
        .drop(columns=['offset', 'str_date']) # Clean up temp columns
        )
    return autos

In [4]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

C:\Users\dsgou\AppData\Local\Temp\ipykernel_48696\1960334968.py:24: DtypeWarning: Columns (69,71,72,73,74,75,77,80) have mixed types. Specify dtype option on import or set low_memory=False.
  raw =  pd.read_csv(url)


,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00


In [ ]:
df.VClass

0            Two Seaters
1            Two Seaters
2        Subcompact Cars
3                   Vans
4           Compact Cars
              ...       
49841       Compact Cars
49842       Compact Cars
49843       Compact Cars
49844       Compact Cars
49845       Compact Cars
Name: VClass, Length: 49846, dtype: object

In [ ]:
# create a column per unique value of df.vClass
pd.get_dummies(df.VClass)

,Compact Cars,Large Cars,Midsize Cars,Midsize Station Wagons,Midsize-Large Station Wagons,Minicompact Cars,Minivan - 2WD,Minivan - 4WD,Small Pickup Trucks,Small Pickup Trucks 2WD,...,Standard Pickup Trucks 4WD,Standard Pickup Trucks/2wd,Standard Sport Utility Vehicle 2WD,Standard Sport Utility Vehicle 4WD,Subcompact Cars,Two Seaters,Vans,Vans Passenger,"Vans, Cargo Type","Vans, Passenger Type"
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49841,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49842,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49843,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
49844,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [5]:
# pick object/str unique types, doesn't make sense for numeric/continuous values
(
    df.select_dtypes(object).nunique().index
)

Index(['make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr', 'evMotor',
       'fuelType', 'trans_dscr'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49846 entries, 0 to 49845
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype                           
---  ------      --------------  -----                           
 0   year        49846 non-null  int64                           
 1   make        49846 non-null  object                          
 2   model       49846 non-null  object                          
 3   trany       49835 non-null  object                          
 4   drive       48660 non-null  object                          
 5   VClass      49846 non-null  object                          
 6   eng_dscr    31637 non-null  object                          
 7   barrels08   49846 non-null  float64                         
 8   city08      49846 non-null  int64                           
 9   comb08      49846 non-null  int64                           
 10  range       49846 non-null  int64                           
 11  evMotor     3447 non-null   

In [14]:
# Deploying one-hot encoding
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest


cat_cols = ['make', 'model', 'trany', 'drive',
        'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr']

# Encoding needs to happen in sequence (after or before the transformer steps)
# Why? because everything within a step is executed parallely (cyl_imputer and displ_imputer)
cat_pipeline = Pipeline([
    ('cat_imputer', cat_imputer),
    ('one_hot_encoder', one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('cat_pipeline', cat_pipeline, cat_cols)
    ],
    remainder='passthrough'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.9685591679801825
mse is 0.04727504673546944


c:\repos\ai-portfolio\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 5, 6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\repos\ai-portfolio\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 5, 6, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [15]:
!pip install category_encoders

In [16]:
# Hash Encoding
high_cardinality_cols = (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s>40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()

low_cardinality_cols =  (
    df.select_dtypes(object)
    .nunique()
    .pipe(lambda s: s[s<40]) # only pick high cardinality values i.e., columns with >40 unique values
    .index
).to_numpy()


high_cardinality_cols=list(high_cardinality_cols)
low_cardinality_cols=list(low_cardinality_cols)

high_cardinality_cols



['make', 'model', 'eng_dscr', 'evMotor', 'trans_dscr']

In [17]:
# Exploring hashencoding
from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        ('hashing_encoder', hashing_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.859361961340446
mse is 0.10430743853708524


Target Encoding transforms categorical labels into a statistical "track record." It maps features into high-signal "buckets" representing the likelihood of producing specific target ranges, effectively pre-calculating the relationship so the model doesn't have to.

To prevent "perfect hindsight," you blend the specific categorical mean with the global target mean (the baseline for all data). This acts as a stabilizer: it pulls noisy, low-volume categories toward the global norm, forcing the model to learn broader patterns rather than simply memorizing flukes.

In [18]:
# Target Encoding
from sklearn.preprocessing import TargetEncoder

tc = TargetEncoder(target_type='continuous', random_state=42)
tc.fit_transform(X_train[['make']], y_train)

array([[2.4586999 ],
       [3.24382207],
       [2.98412361],
       ...,
       [2.80760122],
       [3.01021717],
       [2.80764152]], shape=(37384, 1))

In [19]:
# Exploring target encoding


from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)


preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', tc, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # if we print tmp_X now, function transformer will have stored the intermediate value of X between preprocessor and std_scaler in it
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.9208685194419987
mse is 0.08049456203535894


In [22]:
cat_cols = ['trany', 'drive', 'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr']
high_cardinality_cols=[]
low_cardinality_cols=[]
for col in cat_cols:
    if df[col].nunique() < 40:
        low_cardinality_cols.append(col)
    else:
        high_cardinality_cols.append(col)

low_cardinality_cols

['drive', 'VClass', 'fuelType']

In [26]:
# Exercise

from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

set_config(transform_output='pandas')

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

std_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

preprocessor = ColumnTransformer(
    transformers = [
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', target_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('std_scaler', std_scaler),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)

# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.8128562089399961
mse is 0.1310965076403578


There are specific column level transformations we may need to apply
For eg. if a car is an EV indicated by evMotor and eng_dscr is empty, we may want to add 'EV', if it is not an EV (indicated by some other value in the evMotor column)we may need to add 'FFS'.
Such business-logic dependent transformations require custom transformers, this eliminates the need for a separate data-cleaning step external to the pipeline.

Below is an AI generated snippet to do so

There are specific column level transformations we may need to apply
For eg. if a car is an EV indicated by evMotor and eng_dscr is empty, we may want to add 'EV', if it is not an EV (indicated by some other value in the evMotor column)we may need to add 'FFS'.
Such business-logic dependent transformations require custom transformers, this eliminates the need for a separate data-cleaning step external to the pipeline.

Below is an AI generated snippet to do so

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class EngineDescImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        # Nothing to learn from the data, just return self
        return self

    def transform(self, X):
        # Create a copy to avoid SettingWithCopy warnings
        X = X.copy()

        # Logic: If eng_dscr is missing:
        # Check evMotor. If NaN -> NOTAVL, else -> EV
        mask = X['eng_dscr'].isna()

        X.loc[mask, 'eng_dscr'] = X.loc[mask, 'evMotor'].apply(
            lambda x: 'EV' if pd.notna(x) else 'NOTAVL'
        )

        # Return only the column we need for the next step in the pipeline
        # or the whole dataframe depending on your ColumnTransformer setup
        return X[['eng_dscr']]

**Feature extraction**

Feature extraction is the process of transforming raw data into features that better represent the underlying problem to the predictive models, resulting in improved model accuracy on the unseen data


**PCA**

*Principal Component Analysis (PCA)* is a technique for reducing the dimensionality of data. It can also remove noise and might be useful as feature engineering for oither algorithms.

In [ ]:
# loader scripts

import requests
import pandas as pd
import numpy as np

def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
    .groupby(tz_offset)
    [time_col]
    .transform(lambda s: pd.to_datetime(s)
    .dt.tz_localize(s.name, ambiguous=True)
    .dt.tz_convert(tz_name))
    )



def load_fuel_economy_data() -> pd.DataFrame :
    url = 'https://www.fueleconomy.gov/feg/epadata/vehicles.csv'
    cols = ['year', 'make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr',
            'barrels08', 'city08', 'comb08', 'range', 'evMotor', 'cylinders', 'displ', 'fuelCost08',
            'fuelType', 'highway08', 'trans_dscr','createdOn']

    raw =  pd.read_csv(url)
    autos = (raw
        .loc[:, cols]
        .assign(
            # Extract Timezone Offset (e.g., EDT -> EST5EDT)
            offset=(raw.createdOn.str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                    ['offset'].replace('EDT', 'EST5EDT')),

            # Reconstruct date string (removing the original TZ name for parsing)
            str_date=(raw.createdOn.str.slice(4, 19) + " " +
                    raw.createdOn.str.slice(-4)),

            # Convert to localized datetime
            createdOn=lambda df_: to_tz(df_, 'str_date', 'offset', 'America/New_York')
        )
        .drop(columns=['offset', 'str_date']) # Clean up temp columns
        )
    return autos

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df[0:200]

C:\Users\dsgou\AppData\Local\Temp\ipykernel_52988\1960334968.py:24: DtypeWarning: Columns (69,71,72,73,74,75,77,80) have mixed types. Specify dtype option on import or set low_memory=False.
  raw =  pd.read_csv(url)


,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1993,Ford,Taurus Wagon,Automatic 4-spd,Front-Wheel Drive,Midsize-Large Station Wagons,(FFS),14.167143,18,21,0,NaN,6.0,3.0,2850,Regular,27,NaN,2013-01-01 00:00:00-05:00
196,1993,Ford,Taurus Wagon,Automatic 4-spd,Front-Wheel Drive,Midsize-Large Station Wagons,(FFS),14.875500,17,20,0,NaN,6.0,3.8,3000,Regular,25,NaN,2013-01-01 00:00:00-05:00
197,1993,Mercury,Sable Wagon,Automatic 4-spd,Front-Wheel Drive,Midsize-Large Station Wagons,(FFS),14.167143,18,21,0,NaN,6.0,3.0,2850,Regular,27,NaN,2013-01-01 00:00:00-05:00
198,1993,Mercury,Sable Wagon,Automatic 4-spd,Front-Wheel Drive,Midsize-Large Station Wagons,(FFS),14.875500,17,20,0,NaN,6.0,3.8,3000,Regular,25,NaN,2013-01-01 00:00:00-05:00


['make', 'model', 'eng_dscr', 'evMotor', 'trans_dscr']

In [ ]:
# Exercise

from category_encoders import hashing
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

set_config(transform_output='pandas')

X=df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)

std_scaler = StandardScaler()
cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
disply_imputer = SimpleImputer(strategy='median')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore') # pick only 10 cats, ignore the rest
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True) # for high cardinality values use a hash function to map the entire cardinality to a fixed set of cols (10 here)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

preprocessor = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', disply_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols),
        ('target_encoder', target_encoder, high_cardinality_cols),
    ],
    remainder='drop'
)

# take the X and set it to whatever variable name 'name' you pass to it
def debug_transformer(X, name):
    globals()[name]=X
    return X

# train the model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('std_scaler', std_scaler),
    ('pca', PCA(n_components=10)),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name':'tmp_X'})),
    ('lr', LinearRegression())])


pipeline.fit(X_train, y_train)

# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


R2 score is 0.8901176426240702
mse is 0.09451911308027362


In [ ]:
tmp_X

# This gives the Principal components, but not their labels or any intuitive knowledge about it

,pca0,pca1,pca2,pca3,pca4,pca5,pca6,pca7,pca8,pca9
46169,-0.934654,1.929027,-0.151452,0.430483,-0.997105,-0.432472,-0.044837,-0.010938,0.435070,0.082642
38354,-0.407744,-0.729804,1.087595,-2.521069,-2.657637,1.438482,-0.528011,0.310687,0.098379,-0.413910
25598,-7.008661,-1.242131,3.508653,4.191173,1.281665,-0.922092,0.730382,0.454095,0.314261,-0.222303
5235,2.708720,0.020386,2.011660,-0.349089,0.779829,-1.380802,-0.380158,-0.447244,0.040723,0.273387
10009,-0.304181,1.802653,0.255445,0.377667,-0.452080,-0.861336,0.034588,-0.615280,0.021882,0.223318
...,...,...,...,...,...,...,...,...,...,...
32846,-1.067099,1.897282,-0.241509,0.225825,-1.110904,-0.636531,-0.376183,-0.304225,0.135159,0.095316
35648,-2.011794,1.676733,-0.814877,-0.235925,-0.161411,-0.035565,0.158134,-0.033742,-0.047513,-0.019479
6527,0.467109,-0.514833,-0.398746,-1.220973,2.567271,-1.308289,0.712837,-0.865389,0.355463,-1.344807
31425,1.563415,-1.841377,-1.729130,0.717931,0.570441,0.169959,-0.381711,1.449982,-0.798054,0.644092
